In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [ ]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a4-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}"

In [ ]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

In [ ]:
!ls {PROJECT_ROOT}


In [ ]:
# Install required packgaes
!pip install torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
from torchvision import datasets, transforms
from mnist import Net, SvdLinear, test

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
model.load_state_dict(torch.load("mnist_fc.pt"))

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_dataset = datasets.MNIST(
    'data',
    train=False,
    transform=transform
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000)

In [ ]:
runtimes = []
accuracies = []
compression_ratios = []

fc1_compression_factor_start = 128
fc1_compression_factor_end = 8
fc1_step = 16

fc1_start_rank = model.fc1.weight.size(1) // fc1_compression_factor_start
fc1_end_rank = model.fc1.weight.size(1) // fc1_compression_factor_end

fc2_compression_factor_start = 128
fc2_compression_factor_end = 8
fc2_step = 8

fc2_start_rank = model.fc2.weight.size(1) // fc2_compression_factor_start
fc2_end_rank = model.fc2.weight.size(1) // fc2_compression_factor_end

for rank_fc1 in range(fc1_start_rank, fc1_end_rank + 1, fc1_step):
    for rank_fc2 in range(fc2_start_rank, fc2_end_rank + 1, fc2_step):
        model_for_svd = Net().to(device)
        model_for_svd.load_state_dict(model.state_dict())
        
        model_for_svd.fc1 = SvdLinear(model_for_svd.fc1, rank_fc1)
        model_for_svd.fc2 = SvdLinear(model_for_svd.fc2, rank_fc2)
        
        start = time.time()
        correct_classification = test(
            model_for_svd,
            device,
            test_loader
        )
        end = time.time()
        runtime = end - start
        
        runtimes.append(runtime)
        accuracy = (
            100.0
            * (
                correct_classification
                / len(test_loader.dataset)
            )
        )
        accuracies.append(accuracy)
        
        fc1_original_size = model.fc1.weight.numel()
        fc2_original_size = model.fc2.weight.numel()
        total_original_size = fc1_original_size + fc2_original_size

        fc1_compressed_size = (
            rank_fc1
            * (
                model.fc1.weight.size(1)
                + model.fc1.weight.size(0)
            )
        )
        fc2_compressed_size = (
            rank_fc2
            * (
                model.fc2.weight.size(1)
                + model.fc2.weight.size(0)
            )
        )
        total_compressed_size = (
            fc1_compressed_size
            + fc2_compressed_size
        )

        ratio = total_original_size / total_compressed_size
        compression_ratios.append(ratio)
        
        print(f'Accuracy: {accuracy:.2f}%; Compression Ratio: {ratio:.2f}; Runtime: {runtime:.2f}s')


In [ ]:
# Plot 1: Compression Ratio vs Accuracy
plt.figure(figsize=(8, 6))
plt.scatter(compression_ratios, accuracies)
plt.xlabel('Compression Ratio')
plt.ylabel('Accuracy (%)')
plt.title('Compression Ratio vs Accuracy')
plt.grid(True)
plt.show()

In [ ]:
# Plot 2: Compression Ratio vs Runtime
plt.figure(figsize=(8, 6))
plt.scatter(compression_ratios, runtimes)
plt.xlabel('Compression Ratio')
plt.ylabel('Runtime (s)')
plt.title('Compression Ratio vs Runtime')
plt.grid(True)
plt.show()

In [ ]:
# Plot 3: Runtime vs Accuracy
plt.figure(figsize=(8, 6))
plt.scatter(runtimes, accuracies)
plt.xlabel('Runtime (s)')
plt.ylabel('Accuracy (%)')
plt.title('Runtime vs Accuracy')
plt.grid(True)
plt.show()